# Lightweight TB-Net  - full reproducibility run

One notebook that produces every number in the paper, from raw images to
aggregated tables with confidence intervals.

**Design:** this notebook is a *driver*. All the science lives in versioned
`.py` files in the public repo, which are cloned in cell 1 and invoked as
subprocesses. Nothing is pasted into cells, and no private dataset is
required  - a reviewer can run this as-is, or ignore it and run the same
commands on a desktop.

**Datasets to attach (both public):**
- `tawsifurrahman/tuberculosis-tb-chest-xray-dataset`  - the working cohort
- `kmader/pulmonary-chest-xray-abnormalities`  - Montgomery + Shenzhen, the
  cohorts the original TB-Net was evaluated on (optional but recommended;
  verify its contents before relying on the labels)

**Settings:** GPU T4 x1, Internet **on** (needed for `git clone`).

| Stage | What it settles | ~T4 time |
|---|---|---|
| 0 smoke test | code works before you spend GPU hours | 1 min |
| 1 cache | decode once, ~8 s/epoch instead of ~1.2 min | 4 min |
| 2 audit | which of the two conflicting result tables is real | 5 min |
| 3 baseline | 2 archs x 2 preprocessings x 5 seeds, mean +/- std | 45 min |
| 4 prune | one-shot 25/50/75 **and** iterative, 5 seeds | 60 min |
| 5 distill | distilled **vs. from-scratch control**, 5 seeds | 70 min |
| 6 quantize | FP16 / INT8 / combined + ONNX latency | 15 min |
| 7 external | Montgomery + Shenzhen  - the real comparison | 5 min |
| 8 phone | per-distortion ablation + augmented fine-tune | 45 min |
| 9 summary | mean +/- std tables + environment.json | 1 min |

Total ~ **4-5 h**, inside Kaggle's 12 h session cap. Every stage skips itself
if its `results/*.csv` already exists, so a killed session resumes cheaply
(see the last cell on saving `/kaggle/working` as a dataset).

## 0 - Configuration and repo checkout

In [ ]:
import glob, json, os, subprocess, sys, time

REPO_URL  = "https://github.com/AIscend-Research/tb-repro"
REPO_DIR  = "/kaggle/working/tb-repro"
SEEDS     = [0, 1, 2, 3, 4]      # drop to [0] for a fast dry run
ARCHS     = ["compact", "full"]  # 0.27M re-implementation, ~4.2M reconstruction
CACHES    = ["faithful", "simple"]
SMOKE     = True

# Optional: a previous run saved as a Kaggle dataset, to resume mid-way.
RESUME_FROM = "/kaggle/input/tbnet-repro-results"

ON_KAGGLE = os.path.exists("/kaggle/input")
if not ON_KAGGLE:
    REPO_DIR = os.getcwd()   # running from a local repo checkout

if ON_KAGGLE and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
SRC = os.path.join(REPO_DIR, "src")
print("repo:", REPO_DIR)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip())


def run(*cmd, quiet=False):
    """Run a repo script, streaming its output, and fail loudly."""
    cmd = [sys.executable] + list(cmd)
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        if not quiet:
            print(line, end="")
    p.wait()
    print(f"[{(time.time() - t0) / 60:.1f} min, exit {p.returncode}]")
    if p.returncode != 0:
        raise RuntimeError("".join(tail[-40:]))

In [ ]:
# Record the exact environment  - ReScience reviewers need this to be pinned.
import torch, numpy, pandas, sklearn, cv2

env = {
    "python": sys.version.split()[0], "torch": torch.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "numpy": numpy.__version__, "pandas": pandas.__version__,
    "sklearn": sklearn.__version__, "cv2": cv2.__version__,
}
print(json.dumps(env, indent=2))
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
json.dump(env, open(f"{REPO_DIR}/results/environment.json", "w"), indent=2)

try:
    import onnxruntime
    print("onnxruntime", onnxruntime.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "onnx", "onnxruntime"], check=True)

In [ ]:
# Dataset locations. Defaults are the paths you confirmed; both are searched
# for recursively as a fallback in case Kaggle mounts them elsewhere.
TB_DATASET  = "/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset"
MC_SZ_DATASET = "/kaggle/input/datasets/kmader/pulmonary-chest-xray-abnormalities"


def find_dir(root, must_contain):
    if not os.path.isdir(root):
        return None
    for dirpath, dirnames, _ in os.walk(root):
        if all(m in dirnames for m in must_contain):
            return dirpath
    return None


DATA_PATH = EXTERNAL_PATH = None
if ON_KAGGLE:
    DATA_PATH = find_dir(TB_DATASET, ["Normal", "Tuberculosis"]) \
        or find_dir("/kaggle/input", ["Normal", "Tuberculosis"])
    # Montgomery + Shenzhen: take the common parent of BOTH cohorts, not just
    # whichever one globs first.
    hits = (glob.glob(f"{MC_SZ_DATASET}/**/MCUCXR_*.png", recursive=True)
            + glob.glob(f"{MC_SZ_DATASET}/**/CHNCXR_*.png", recursive=True)
            or glob.glob("/kaggle/input/**/MCUCXR_*.png", recursive=True)
            + glob.glob("/kaggle/input/**/CHNCXR_*.png", recursive=True))
    if hits:
        EXTERNAL_PATH = os.path.commonpath([os.path.dirname(h) for h in hits])
else:
    DATA_PATH = os.path.join(REPO_DIR, "data")

assert DATA_PATH, f"TB dataset not found under {TB_DATASET}"
n_imgs = len(glob.glob(os.path.join(DATA_PATH, "**", "*.png"), recursive=True))
print(f"DATA_PATH     = {DATA_PATH}  ({n_imgs} png)")
print(f"EXTERNAL_PATH = {EXTERNAL_PATH}")
if EXTERNAL_PATH:
    # exclude the lung-segmentation masks, which share the CXR filenames
    nomask = lambda ps: [p for p in ps if "mask" not in p.lower()]
    n_mc = len(nomask(glob.glob(f"{EXTERNAL_PATH}/**/MCUCXR_*.png", recursive=True)))
    n_sz = len(nomask(glob.glob(f"{EXTERNAL_PATH}/**/CHNCXR_*.png", recursive=True)))
    print(f"  Montgomery={n_mc}  Shenzhen={n_sz}")

# Resume: copy forward results/checkpoints/cache from a previous session.
if os.path.isdir(RESUME_FROM):
    import shutil
    for sub in ("results", "checkpoints", "cache"):
        src = os.path.join(RESUME_FROM, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(REPO_DIR, sub), dirs_exist_ok=True)
            print(f"resumed {sub}/ from {RESUME_FROM}")

## Stage 0b - Regenerate the splits  - **do not skip this**

The splits committed to `data_splits/` are contaminated. They list 1,400 TB
entries drawn from only **700 distinct images**, present twice under two
names (`Tuberculosis-N.png` and `TB-N.png`). Because the split was taken over
rows rather than distinct images, **222 of the 700 TB images sit in both the
training set and a held-out set**, and 115 of the 140 TB images in the old
`test.csv` are duplicates of training images.

Every metric in the paper and in `results_clean.csv` was computed on that
leak, which is the most likely explanation for the near-perfect scores.

The cell below regenerates splits keyed on a canonical image identity, then
asserts no image crosses a split boundary. The real class ratio is **5:1**
(3,500 normal : 700 TB), not the 2.5:1 the paper analyses in section 5.3  - that
section needs rewriting too.

In [ ]:
run(f"{SRC}/make_splits.py", "--data-path", DATA_PATH,
    "--out", f"{REPO_DIR}/data_splits", "--seed", "42")
run(f"{SRC}/make_splits.py", "--verify-only", "--out", f"{REPO_DIR}/data_splits")

## Stage 0 - Smoke test (~1 min)

Runs every stage on 60 synthetic images at 1 epoch. If this fails, fix the
code before spending GPU hours.

In [ ]:
if SMOKE:
    run(f"{SRC}/smoke_test.py")

## Stage 1 - Build the image cache (~4 min, one time)

Two variants, so the preprocessing pipeline itself becomes an ablation:

- **faithful**  - the paper's B-channel split, padding-aware auto-crop, DSI
  crop and corner masking (`src/preprocessing.py` + `_tf1_reference/dsi.py`).
  This pipeline exists in the repo but *nothing currently imports it*.
- **simple**  - grayscale + resize, which is what `src/train.py` actually did.

The gap between the two is a publishable result on its own.

In [ ]:
if not os.path.exists(f"{REPO_DIR}/cache/faithful.npy"):
    run(f"{SRC}/build_cache.py", "--data-path", DATA_PATH, "--variant", "both")
else:
    print("cache already built")

# Sanity-check the two pipelines visually before trusting 5 hours of training.
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for r, variant in enumerate(CACHES):
    arr = np.load(f"{REPO_DIR}/cache/{variant}.npy", mmap_mode="r")
    for c in range(4):
        axes[r, c].imshow(arr[c * 137], cmap="gray", vmin=0, vmax=255)
        axes[r, c].set_title(f"{variant}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout(); plt.show()

## Stage 3 - Baselines: 2 archs x 2 caches x 5 seeds (~45 min)

Replaces every single-run number with mean +/- std. `full` is a **capacity-
matched reconstruction**, not a faithful port  - the original graph ships
inside `model_train.meta`, whose download is dead, so a layer-for-layer port
is impossible from released materials. Say so in the paper.

In [ ]:
run(f"{SRC}/models_repro.py")   # print parameter counts for the paper
run(f"{SRC}/experiments.py", "--stage", "baseline",
    "--arch", *ARCHS, "--caches", *CACHES,
    "--seeds", *map(str, SEEDS), "--epochs", "10")

In [ ]:
# Optional: the class-weighted + cosine variant that paper section 5.9 only predicted.
run(f"{SRC}/experiments.py", "--stage", "baseline", "--force",
    "--arch", ARCHS[0], "--caches", "faithful", "--weighted", "1",
    "--cosine", "--seeds", *map(str, SEEDS), "--epochs", "10")

## Stage 4 - Pruning, one-shot and iterative (~60 min)

section 4.3 speculates that a gradual schedule would recover the 75 % sensitivity
cliff better than one-shot. This measures it instead of speculating.

In [ ]:
run(f"{SRC}/experiments.py", "--stage", "prune", "--arch", *ARCHS,
    "--caches", "faithful", "--seeds", *map(str, SEEDS),
    "--ft-epochs", "5", "--iter-steps", "8", "--iter-amount", "0.16")

## Stage 5 - Distillation **with a from-scratch control** (~70 min)

Without the control you cannot claim distillation helped  - the student might
reach the same place on hard labels alone.

In [ ]:
run(f"{SRC}/experiments.py", "--stage", "distill", "--arch", ARCHS[0],
    "--caches", "faithful", "--seeds", *map(str, SEEDS),
    "--distill-epochs", "15")

## Stage 6 - Quantization, combined compression, ONNX latency (~15 min)

Includes the pruned-25 % + FP16 combination that section 5.6 recommends and cites a
`combined_compress.py` for  - a file that does not exist in the repo. Latency
is a median of 100 CPU runs with IQR; on a shared Kaggle vCPU report it as
indicative, not as device latency.

In [ ]:
run(f"{SRC}/experiments.py", "--stage", "quantize", "--arch", *ARCHS,
    "--caches", "faithful", "--seeds", *map(str, SEEDS))

## Stage 7 - External validation on Montgomery + Shenzhen (~5 min)

The only comparison that licenses a delta-vs-original table: these are the
cohorts Wong et al. actually reported on. Labels come from the filename
suffix (`*_0` normal, `*_1` TB)  - verify that on a handful of files first.

In [ ]:
if EXTERNAL_PATH and os.path.isdir(EXTERNAL_PATH):
    run(f"{SRC}/experiments.py", "--stage", "external",
        "--external-path", EXTERNAL_PATH, "--arch", *ARCHS,
        "--caches", "faithful", "--seeds", *map(str, SEEDS))
else:
    print("Montgomery/Shenzhen not attached  - skipping. Attach "
          "kmader/pulmonary-chest-xray-abnormalities to enable this stage.")

## Stage 7b - Cohort overlap check (~3 min)

Does the "external" set actually sit outside the training cohort? The Rahman
database aggregates several sources and the NLM Montgomery/Shenzhen sets are
among them, so this has to be measured rather than assumed. Perceptual hashing
(dHash, Hamming distance <= 5) against every cached training image.

In [ ]:
if EXTERNAL_PATH and os.path.isdir(EXTERNAL_PATH):
    run(f"{SRC}/experiments.py", "--stage", "overlap",
        "--external-path", EXTERNAL_PATH, "--caches", "faithful")

## Stage 8 - Phone-capture robustness (~45 min)

Three things the current extension is missing: a fixed perturbation seed, a
per-distortion ablation (which one causes the collapse?), and an attempted
fix (fine-tuning with the distortions on). Reporting a failure with no
attempted remedy is a weak result.

In [ ]:
run(f"{SRC}/experiments.py", "--stage", "phone", "--arch", *ARCHS,
    "--caches", "faithful", "--seeds", *map(str, SEEDS),
    "--phone-finetune", "--ft-epochs", "5")

## Stage 9 - Aggregate every table (~1 min)

In [ ]:
run(f"{SRC}/experiments.py", "--stage", "summary")

import pandas as pd
for f in sorted(glob.glob(f"{REPO_DIR}/results/summary_*.csv")):
    print(f"\n===== {os.path.basename(f)} =====")
    display(pd.read_csv(f))

## Stage 10 - Analysis and figures

`analyze.py` needs only the standard library; `make_figures.py` needs
matplotlib.

In [ ]:
run(f"{SRC}/analyze.py")
run(f"{SRC}/make_figures.py")

from IPython.display import Markdown, display
display(Markdown(open(f"{REPO_DIR}/results/ANALYSIS.md").read()))

## Package the run

Everything a reviewer needs: per-run CSVs, aggregated tables, the pinned
environment, and the commit the code came from.

To resume a killed session: **Save Version -> Save Output**, publish
`/kaggle/working` as a dataset named `tbnet-repro-results`, attach it to the
next run, and the `RESUME_FROM` cell copies `results/`, `checkpoints/` and
`cache/` forward so completed stages skip themselves.

In [ ]:
import shutil

OUT = "/kaggle/working/artifacts" if ON_KAGGLE else f"{REPO_DIR}/artifacts"
os.makedirs(OUT, exist_ok=True)
for sub in ("results", "checkpoints", "deploy_repro", "figures", "data_splits"):
    src = os.path.join(REPO_DIR, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(OUT, sub), dirs_exist_ok=True)

commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
json.dump({"commit": commit, "repo": REPO_URL, "seeds": SEEDS,
           "archs": ARCHS, "caches": CACHES, "data_path": DATA_PATH,
           "external_path": EXTERNAL_PATH, "env": env},
          open(os.path.join(OUT, "run_manifest.json"), "w"), indent=2)

shutil.make_archive(OUT, "zip", OUT)
print(f"packaged -> {OUT}.zip")
for f in sorted(glob.glob(f"{OUT}/results/*.csv")):
    print(" ", os.path.basename(f), sum(1 for _ in open(f)) - 1, "rows")